# 🏡 Price Prediction Module

Serve the deployed model as an interactive estimator. The user gives a **small set of meaningful
inputs**; we derive the rest and assemble the model's 24 features.

**Current deployed model:** LightGBM, **11.39% MAPE (R² 0.91)** on a held-out test set — a
60/20/20 train/validation/test split where the winning model family is chosen on validation and
test is scored exactly once.

**Reduced input set (user-provided):** property type, sector, area, bedrooms, bathrooms,
furnishing, age/possession, covered parking, and a few amenity toggles (AC, power backup, pool,
corner).

**Everything else is derived (not asked):**
- the four `dist_to_*` → **median distance for that sector** from the training data (all 131
  sectors, matches training). These are a *geographic fact* of the sector the user already picked,
  not a guess.
- `total_floor` (building height, a mid-strength feature) → **asked for flats & builder floors**;
  for **independent houses** it's not applicable, so it defaults to the per-type value (ground).
- `floornum_category` (which floor the unit is on) and `balcony` → **per-property-type defaults**.
  Both are weak features, so we don't ask a second floor question for near-zero accuracy gain.
- `facing` → `'unknown'` (a real training category; facing is a weak driver, not worth asking),
  `open_parking` = 0, `ov_main_road` / `ov_others` = 0.

**What happened to `society`?** It used to be derived here (the sector's most common society) — and
that turned out to be a mistake. Guessing it was correct only ~31% of the time, and serving the
model a wrong-but-confident society made production MAPE **worse** (14.29% vs 11.18% without it,
as measured under the setup in place at the time) — textbook **train/serve skew**. It was dropped
from the model entirely. See `notebooks/error_analysis/society_ablation.ipynb` for the experiment.

Two small artifacts are precomputed: `data/price_prediction/sector_reference.csv` and
`type_reference.csv`.

In [ ]:
import os
import numpy as np
import pandas as pd
import joblib

pd.set_option('display.max_columns', None)
OUT = '../../data/price_prediction'
os.makedirs(OUT, exist_ok=True)

df = pd.read_csv('../../data/fs/feature_selected_properties.csv')
bundle = joblib.load('../../artifacts/best_model.joblib')
pipeline = bundle['pipeline']
residual_q = bundle.get('residual_quantiles', {})
TARGET = 'price_in_cr'
FEATURES = [c for c in df.columns if c != TARGET]
DIST = ['dist_to_cyber_city', 'dist_to_golf_road', 'dist_to_airport', 'dist_to_manesar']
print('model:', bundle['model_name'], '| test MAPE %:', bundle['test_mape_percent'])

## 1. Reference tables

**Sector reference** — median landmark distances per sector (the `society` column is gone; it is no
longer a model feature).
**Type reference** — per-property-type defaults for the fields we no longer ask the user
(`total_floor`, `floornum_category`, `balcony`). This is what makes independent houses behave
correctly without a floor input.

Both files are read at runtime by `api/inference.py`, which is now the single place that turns
user inputs into a price.

In [ ]:
# Median landmark distances per sector. No society column any more — the model
# doesn't take society, so there is nothing to look up for it.
sector_ref = df.groupby('sector').agg(**{d: (d, 'median') for d in DIST}).reset_index()
sector_ref.to_csv(f'{OUT}/sector_reference.csv', index=False)

type_ref = df.groupby('property_type').agg(
    total_floor=('total_floor', 'median'),
    balcony=('balcony', 'median'),
    floornum_category=('floornum_category', lambda x: x.mode().iloc[0]),
).reset_index()
type_ref.to_csv(f'{OUT}/type_reference.csv', index=False)

print('sector_reference:', sector_ref.shape, '| type_reference:')
print(type_ref.to_string(index=False))

> **Conclusion.** Note the per-type defaults: independent houses default to low floors /
> low-rise (correct — they have no building floors), while flats default to a taller building.
> The user never has to enter floor info.

## 2. `build_input_row` — assemble the 24 features from the reduced inputs

This mirrors `api/inference.py`, which is what actually runs in production. The notebook version
exists to *validate* the logic; the API owns it.

In [ ]:
CONST = dict(facing='unknown', open_parking=0.0, ov_main_road=0, ov_others=0)

def build_input_row(inp: dict, sector_ref: pd.DataFrame, type_ref: pd.DataFrame) -> pd.DataFrame:
    pt = inp['property_type']
    sr = sector_ref.loc[sector_ref['sector'] == inp['sector']]
    tr = type_ref.loc[type_ref['property_type'] == pt]
    tr = tr.iloc[0] if not tr.empty else type_ref.iloc[0]

    covered = float(inp.get('covered_parking', 1))
    # total_floor: use the user value if given (flats/builder floors), else per-type default (houses)
    total_floor = float(inp['total_floor']) if inp.get('total_floor') is not None else float(tr['total_floor'])
    row = {
        'area': float(inp['area']),
        'total_floor': total_floor,
        'bathroom': int(inp['bathroom']),
        'property_type': pt,
        'covered_parking': covered,
        'open_parking': CONST['open_parking'],
        'sector': inp['sector'],
        'bedRoom': int(inp['bedRoom']),
        'furnishing': inp.get('furnishing', 'semi-furnished'),
        'balcony': float(tr['balcony']),                            # per-type default
        'age_possession_category': inp.get('age_possession_category', 'New Property'),
        'facing': CONST['facing'],                                  # weak feature -> not asked
        'floornum_category': tr['floornum_category'],               # per-type default
        'total_parking': covered + CONST['open_parking'],
    }
    for flag in ['has_ac', 'has_power_backup', 'has_pool', 'is_corner']:
        row[flag] = int(inp.get(flag, 0))
    row['ov_main_road'] = CONST['ov_main_road']
    row['ov_others'] = CONST['ov_others']
    for d in DIST:
        row[d] = float(sr[d].iloc[0]) if not sr.empty else float(df[d].median())
    return pd.DataFrame([row])[FEATURES]

# a flat (user gives total_floor=25) and an independent house (total_floor omitted -> per-type default)
flat = build_input_row(dict(property_type='Flat', sector='sector 49', area=1600, bedRoom=3,
                            bathroom=3, furnishing='semi-furnished', age_possession_category='New Property',
                            covered_parking=1, total_floor=25, has_ac=1, has_power_backup=1), sector_ref, type_ref)
house = build_input_row(dict(property_type='Independent House', sector='sector 49', area=2500,
                             bedRoom=4, bathroom=4, furnishing='unfurnished',
                             age_possession_category='Old Property', covered_parking=2), sector_ref, type_ref)
print('columns match model:', list(flat.columns) == FEATURES, '| n features:', len(FEATURES))
print('flat  (user floors) -> total_floor', flat['total_floor'].iloc[0], '| floornum_category', flat['floornum_category'].iloc[0])
print('house (no floors)   -> total_floor', house['total_floor'].iloc[0], '| floornum_category', house['floornum_category'].iloc[0])

## 3. Predict + 90% range

In [ ]:
def predict(pipeline, input_df, residual_q):
    price = float(np.expm1(pipeline.predict(input_df)[0]))
    lo = price * np.exp(residual_q['q05']) if residual_q else price * 0.8
    hi = price * np.exp(residual_q['q95']) if residual_q else price * 1.2
    return round(price, 2), round(lo, 2), round(hi, 2)

for name, row in [('Flat 1600sqft 3BHK', flat), ('House 2500sqft 4BHK', house)]:
    p, lo, hi = predict(pipeline, row, residual_q)
    print(f'{name:22s}: ₹{p} Cr  (90% ₹{lo} – ₹{hi} Cr)')

> **Conclusion.** Both property types price sensibly with no floor input required — the
> independent house used its low-rise / ground-floor defaults automatically.

## 4. Validate — model reproduces known listings

Feed 500 real listings' exact features through the pipeline and compare to actual price. Confirms
the serving path (log inverse, column order) is correct.

In [ ]:
sample = df.sample(500, random_state=1)
pred = np.expm1(pipeline.predict(sample[FEATURES]))
actual = sample[TARGET].values
mape = np.mean(np.abs(pred - actual) / actual) * 100
print(f'reconstruction MAPE: {mape:.2f}%  |  within +/-20%: {np.mean(np.abs(pred-actual)/actual<=0.20)*100:.0f}%')
pd.DataFrame({'actual_cr': actual[:5].round(2), 'predicted_cr': pred[:5].round(2)})

> **Conclusion.** Serving path is correct. Reducing the inputs trades a little granularity
> (notably `total_floor`, a mid-strength feature, now uses a per-type default) for a much simpler
> form — a deliberate UX choice.

---
### Summary
A streamlined estimator: ~10 meaningful inputs, everything else derived from `sector_reference`
and `type_reference`. Independent houses are handled correctly without asking for floors.

**Where this logic actually lives:** `api/inference.py`. The FastAPI service owns the
"inputs → 24 features → price" path and `pages/Price_Prediction.py` calls that API, so the number
a user sees and the number the API returns cannot drift apart. This notebook validates the logic;
it does not run in production.

**The `society` lesson.** This module originally derived `society` from the sector because the model
demanded it. Measuring that guess honestly (~31% accurate → 14.29% production MAPE vs 11.18%
without) showed the feature was a liability, and it was removed from the model. The principle:
**never train on a feature you can't obtain honestly at serving time.**